In [129]:
import pandas as pd
import os
import geopandas as gpd
from shapely import Polygon
from statistics import mean

In [145]:
data = {
    "nazwa warstwy": [],
    "rodzaj geometrii": [],
    "ilosc obiektow": [],
    "srednia dlugosc obiektow": [],
    "srednia zlozonosc linii": [],
    "srednie wypelnienie przestrzeni": []
}
informacje = pd.DataFrame(data)

In [59]:
directory = 'dane'
nazwy = []
pliki = []

for root, _, files in os.walk(directory):  
    for filename in files:
        if filename.endswith('.shp'):
            nazwy.append(filename[-10:-4])
            i = gpd.read_file(os.path.join(root, filename))
            i.to_crs("EPSG:2180", inplace=True)
            pliki.append(i)

In [96]:
#pliki[0].head()

In [146]:
informacje["nazwa warstwy"] = nazwy
informacje["rodzaj geometrii"] = [pd.unique(i.geometry.type)[0] for i in pliki]
informacje["ilosc obiektow"] = [i.geometry.count() for i in pliki]
informacje["srednia dlugosc obiektow"] = [i.geometry.length.mean() if not pd.unique(i.geometry.type) == 'Point' and not pd.unique(i.geometry.type) == 'MultiPoint' else "Nie dotyczy" for i in pliki]

In [147]:
zlozonosc = []

for i in pliki:
    if pd.unique(i.geometry.type) == 'Point' or pd.unique(i.geometry.type) == 'MultiPoint':
        zlozonosc.append("Nie dotyczy")
    else:
        zlozonosc.append(i.geometry.count_coordinates().mean()/i.geometry.length.mean())

informacje["srednia zlozonosc linii"] = zlozonosc

In [ ]:
bb_mean = []
wypelnienie = []

for i in pliki:
    if pd.unique(i.geometry.type) == 'Polygon' or pd.unique(i.geometry.type) == 'MultiPolygon':
        bb_mean = []
        for j in range(i.geometry.count()):
            bb = Polygon([(i.iloc[j].geometry.bounds[0], i.iloc[j].geometry.bounds[1]), (i.iloc[j].geometry.bounds[2], i.iloc[j].geometry.bounds[1]), (i.iloc[j].geometry.bounds[2], i.iloc[j].geometry.bounds[3]), (i.iloc[j].geometry.bounds[0], i.iloc[j].geometry.bounds[3])])
            bb_mean.append(bb.area)
        wypelnienie.append(i.geometry.area.mean()/mean(bb_mean))
    else:
        wypelnienie.append("Nie dotyczy")

In [148]:
informacje["srednie wypelnienie przestrzeni"] = wypelnienie

In [149]:
informacje.head(10)

,nazwa warstwy,rodzaj geometrii,ilosc obiektow,srednia dlugosc obiektow,srednia zlozonosc linii,srednie wypelnienie przestrzeni
0,ADJA_A,Polygon,13,144045.831761,0.021115,0.51258
1,ADMS_A,Polygon,262,10342.933677,0.023395,0.496708
2,ADMS_P,Point,262,Nie dotyczy,Nie dotyczy,Nie dotyczy
3,BUBD_A,Polygon,66907,47.852393,0.171805,0.558707
4,BUHD_A,Polygon,5,344.96176,0.065515,0.404016
5,BUHD_L,LineString,31,8.020859,0.349894,Nie dotyczy
6,BUIB_A,Polygon,21,398.595557,0.016964,0.073804
7,BUIB_L,LineString,10,211.517356,0.01371,Nie dotyczy
8,BUIN_L,LineString,185,37.174688,0.094514,Nie dotyczy
9,BUIT_A,Polygon,8,268.971053,0.038108,0.642587


In [150]:
informacje.to_csv("out/informacje.csv", encoding="utf-8", index=False)